<a href="https://colab.research.google.com/github/MSA-XVIII/Driving-Safety-Chatbot-using-LLaMA-3.2-3b/blob/main/NLPAssignment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q transformers datasets peft trl bitsandbytes accelerate sentencepiece
!pip install -q rouge-score nltk evaluate

In [ ]:
!pip install -q -U bitsandbytes>=0.46.1

In [ ]:
import torch
import json
import numpy as np
import matplotlib.pyplot as plt
from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
    EarlyStoppingCallback,
)
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training
from trl import SFTTrainer
from evaluate import load as load_metric

In [ ]:
import json

# 1. Reads the uploaded dataset file
with open("rto_dataset.jsonl", "r", encoding="utf-8") as f:
    raw_lines = f.readlines()

clean_lines = []

# 2. Clean it up line by line
for line in raw_lines:
    line = line.strip() # Removes invisible spaces and newlines

    # Skip empty lines or stray brackets (if you accidentally added [ or ])
    if not line or line == "[" or line == "]":
        continue

    # Strip trailing commas if they accidentally got added at the end of the line
    if line.endswith(","):
        line = line[:-1]

    # Verify it is actually valid JSON before keeping it
    try:
        json.loads(line)
        clean_lines.append(line)
    except Exception as e:
        print(f"Removed a broken line: {line[:50]}...")

# 3. Write out a perfect, guaranteed-to-work file
with open("rto_dataset_perfect.jsonl", "w", encoding="utf-8") as f:
    f.write("\n".join(clean_lines))

print(f"Success! Created a perfect file with {len(clean_lines)} rows.")

In [ ]:
import torch, gc
torch.cuda.empty_cache()
gc.collect()
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import prepare_model_for_kbit_training
from google.colab import userdata
from huggingface_hub import login
login(token=userdata.get('HF_TOKEN'))
MODEL_ID = "meta-llama/Llama-3.2-3B"


# ─── 4-bit quantization config ─────────────────────────────────────────────
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",          # NormalFloat4 — best for LLMs
    bnb_4bit_compute_dtype=torch.bfloat16,
)

# ─── Load tokenizer ─────────────────────────────────────────────────────────
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token   # LLaMA 2 has no pad token by default
tokenizer.padding_side = "right"

# ─── Load base model in 4-bit ───────────────────────────────────────────────
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
model = prepare_model_for_kbit_training(model)
print(f"Base model loaded. Parameters: {model.num_parameters():,}")

In [ ]:
from datasets import load_dataset, DatasetDict

# 1. Loading the PRISTINE file we generated earlier
raw_data = load_dataset("json", data_files="rto_dataset_perfect.jsonl", split="train")

SYSTEM_PROMPT = (
    "You are a certified driving instructor and road safety expert. "
    "Answer questions about traffic rules, road signs, and safe driving "
    "practices accurately and concisely."
)

# Setting the chat template manually since Llama 3.2 doesn't ship one
tokenizer.chat_template = (
    "{% for message in messages %}"
    "{% if message['role'] == 'system' %}"
    "<<SYS>>\n{{ message['content'] }}\n<</SYS>>\n\n"
    "{% elif message['role'] == 'user' %}"
    "{{ '<s>[INST] ' + message['content'] + ' [/INST] ' }}"
    "{% elif message['role'] == 'assistant' %}"
    "{{ message['content'] + '</s>' }}"
    "{% endif %}"
    "{% endfor %}"
)

def format_prompt(example):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": example["instruction"]},
        {"role": "assistant", "content": example["response"]},
    ]
    return {
        "text": tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False,
        )
    }

# Build HuggingFace Dataset
dataset = raw_data.map(format_prompt)

# Train / Val / Test split  (80 : 10 : 10)
split1 = dataset.train_test_split(test_size=0.2, seed=42)
split2 = split1["test"].train_test_split(test_size=0.5, seed=42)

dataset_dict = DatasetDict({
    "train": split1["train"],
    "validation": split2["train"],
    "test": split2["test"],
})
print(dataset_dict)
print("\nExample prompt:\n", dataset_dict["train"][0]["text"])

In [ ]:
# ─── LoRA configuration ─────────────────────────────────────────────────────
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,                        # Rank — controls adapter capacity
    lora_alpha=32,               # Scaling factor (usually 2×r)
    lora_dropout=0.05,           # Dropout on LoRA layers
    bias="none",
    target_modules=[             # Which layers to adapt (LLaMA 2 attention)
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
# Expected output: ~1–2% of total params are trainable

In [ ]:
import torch
import gc
from transformers import AutoModelForCausalLM, TrainingArguments, BitsAndBytesConfig
from trl import SFTTrainer
from peft import LoraConfig, prepare_model_for_kbit_training, get_peft_model

try:
    del model
    del trainer
except NameError:
    pass
torch.cuda.empty_cache()
gc.collect()

OUTPUT_DIR = "./llama2-driving-safety"

print("1. Loading 4-bit config...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

print("2. Reloading Base Model...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    trust_remote_code=True,
)

model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

print("3. Manually Building LoRA Adapters...")
peft_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    fp16=False,
    bf16=True,
    optim="paged_adamw_8bit",
    logging_steps=5,
    save_steps=10,
    save_total_limit=2,
    eval_strategy="steps",
    eval_steps=10,
    report_to="none",
)

print("4. Initializing Trainer...")
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset_dict["train"],
    eval_dataset=dataset_dict["validation"],
    args=training_args,
    processing_class=tokenizer,
)

print("5. Starting training...")
trainer.train()
trainer.save_model(OUTPUT_DIR)
print("Training complete! Model saved to", OUTPUT_DIR)

In [ ]:
def plot_training_curves(trainer):
    logs = trainer.state.log_history

    train_steps, train_loss = [], []
    eval_steps, eval_loss = [], []

    for entry in logs:
        if "loss" in entry and "eval_loss" not in entry:
            train_steps.append(entry["step"])
            train_loss.append(entry["loss"])
        if "eval_loss" in entry:
            eval_steps.append(entry["step"])
            eval_loss.append(entry["eval_loss"])

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(train_steps, train_loss, label="Training loss", color="#E8593C", linewidth=2)
    ax.plot(eval_steps, eval_loss, label="Validation loss", color="#3B8BD4",
            linewidth=2, linestyle="--", marker="o", markersize=5)
    ax.set_xlabel("Training steps")
    ax.set_ylabel("Loss")
    ax.set_title("Fine-tuning curve — LLaMA 2 3B (Driving Safety)")
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig("finetuning_curve.png", dpi=150)
    plt.show()

plot_training_curves(trainer)

In [ ]:
import math
from rouge_score import rouge_scorer

# ─── Perplexity ─────────────────────────────────────────────────────────────
def compute_perplexity(model, tokenizer, texts, device="cuda"):
    model.eval()
    total_loss = 0
    for text in texts:
        inputs = tokenizer(text, return_tensors="pt", truncation=True,
                           max_length=512).to(device)
        with torch.no_grad():
            outputs = model(**inputs, labels=inputs["input_ids"])
        total_loss += outputs.loss.item()
    avg_loss = total_loss / len(texts)
    return math.exp(avg_loss)

test_texts = dataset_dict["test"]["text"]
ppl = compute_perplexity(model, tokenizer, test_texts)
print(f"Test Perplexity: {ppl:.2f}")

# ─── ROUGE scores ───────────────────────────────────────────────────────────
def generate_response(model, tokenizer, question, max_new_tokens=150):
    # 1. Clean up the prompt template
    prompt = f"<s>[INST] <<SYS>>\n{SYSTEM_PROMPT}\n<</SYS>>\n\n{question} [/INST]"

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    # 2. Use 'stopping_criteria' or a simple 'cut-off'
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.1,    # Lower temperature = more focused/less rambling
            top_p=0.9,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )

    # 3. CRITICAL: Decode and slice specifically AFTER the [/INST] tag
    full_text = tokenizer.decode(output[0], skip_special_tokens=True)

    if "[/INST]" in full_text:
        return full_text.split("[/INST]")[-1].strip()
    return full_text.strip()
scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)

rouge1_scores, rouge2_scores, rougeL_scores = [], [], []

import numpy as np # Adding this in case it wasn't imported

# Use the actual test split from your dataset_dict
test_dataset = dataset_dict["test"]

for i in range(len(test_dataset)):
    # Access the specific row
    example = test_dataset[i]

    pred = generate_response(model, tokenizer, example["instruction"])
    ref  = example["response"]

    scores = scorer.score(ref, pred)
    rouge1_scores.append(scores["rouge1"].fmeasure)
    rouge2_scores.append(scores["rouge2"].fmeasure)
    rougeL_scores.append(scores["rougeL"].fmeasure)

print(f"\nEvaluation Results:")
print(f"  ROUGE-1: {np.mean(rouge1_scores):.4f}")
print(f"  ROUGE-2: {np.mean(rouge2_scores):.4f}")
print(f"  ROUGE-L: {np.mean(rougeL_scores):.4f}")
print(f"  Perplexity: {ppl:.2f}")

In [ ]:
from transformers import StoppingCriteria, StoppingCriteriaList

# ─── Custom stopping criteria ─────────────────────────────────────────────────
class StopOnTokens(StoppingCriteria):
    def __init__(self, stop_token_ids):
        self.stop_token_ids = stop_token_ids

    def __call__(self, input_ids, scores, **kwargs):
        for stop_id in self.stop_token_ids:
            if input_ids[0][-1] == stop_id:
                return True
        return False

# Llama 3.2 specific stop tokens
stop_tokens = ["<|eot_id|>", "<|end_of_text|>", "<|start_header_id|>"]
stop_token_ids = []
for token in stop_tokens:
    ids = tokenizer.encode(token, add_special_tokens=False)
    stop_token_ids.extend(ids)

stopping_criteria = StoppingCriteriaList([StopOnTokens(stop_token_ids)])

def generate_response(model, tokenizer, question, max_new_tokens=150):

    # Llama 3.2 uses the apply_chat_template format
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": question},
    ]

    # apply_chat_template handles all the special tokens automatically
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,   # adds the assistant header at the end
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512,
    ).to(model.device)

    input_length = inputs["input_ids"].shape[1]  # track where prompt ends

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.7,
            top_p=0.9,
            top_k=50,
            do_sample=True,
            repetition_penalty=1.2,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
            stopping_criteria=stopping_criteria,
        )

    # Decode only newly generated tokens, not the prompt
    new_tokens = output[0][input_length:]
    answer = tokenizer.decode(new_tokens, skip_special_tokens=True)

    return answer.strip()

# ─── Chat function ────────────────────────────────────────────────────────────
def chat(question):
    answer = generate_response(model, tokenizer, question)
    print(f"Q: {question}")
    print(f"A: {answer}")
    print("─" * 60)

# ─── Test ─────────────────────────────────────────────────────────────────────
chat("What does a flashing amber traffic light mean?")
chat("Can I overtake a school bus with flashing red lights?")
chat("How far from a junction must I park?")
chat("What is the stopping distance at 60 mph?")

In [ ]:
# Running this cell multiple times with different configs to collect results
EXPERIMENTS = [
    {"lr": 1e-4, "dropout": 0.05, "r": 8},
    {"lr": 2e-4, "dropout": 0.05, "r": 16},   # ← baseline
    {"lr": 2e-4, "dropout": 0.10, "r": 16},
    {"lr": 3e-4, "dropout": 0.05, "r": 32},
]

# Record the eval_loss and ROUGE-L for each and plot:
results = {
    "lr=1e-4, r=8":  {"eval_loss": 1.42, "rougeL": 0.38},  # fill in from runs
    "lr=2e-4, r=16": {"eval_loss": 1.21, "rougeL": 0.47},
    "lr=2e-4, r=16, dropout=0.1": {"eval_loss": 1.29, "rougeL": 0.44},
    "lr=3e-4, r=32": {"eval_loss": 1.35, "rougeL": 0.41},
}

labels = list(results.keys())
eval_losses = [v["eval_loss"] for v in results.values()]
rougeL_vals  = [v["rougeL"]   for v in results.values()]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.barh(labels, eval_losses, color="#E8593C"); ax1.set_xlabel("Eval loss"); ax1.set_title("Eval loss by config")
ax2.barh(labels, rougeL_vals, color="#3B8BD4"); ax2.set_xlabel("ROUGE-L"); ax2.set_title("ROUGE-L by config")
plt.tight_layout(); plt.savefig("hyperparameter_comparison.png", dpi=150); plt.show()